# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 06: Multivariate Time-Series Anomaly Detection from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In mission-critical infrastructure (data center servers, jet engines, power grids):
- Thousands of telemetry sensors generate multivariate time-series metrics simultaneously.
- Catastrophic failures are extremely rare (labeled anomaly data is $< 0.01\%$).

**Unsupervised Anomaly Detection via Reconstruction Error:**
1. Train a model on normal operational telemetry.
2. The model learns the typical correlation patterns across sensors.
3. When an anomalous event occurs (e.g. pressure spike preceding a leak), the model fails to reconstruct the unusual pattern, producing an **Anomaly Score Spike**!

---

### 2. Mathematical Foundations: Rolling Reconstruction Error
Given input window $X_t \in \mathbb{R}^{W \times D}$ and reconstructed window $\hat{X}_t$:

$$\text{Error}(t) = \frac{1}{D} \sum_{d=1}^D (X_{t, d} - \hat{X}_{t, d})^2$$

#### Dynamic Adaptive Thresholding:
Static thresholds fail because baseline metrics exhibit day/night seasonal cycles.
We compute a **Rolling Z-Score Threshold**:
$$\tau_t = \mu_{\text{error}}(t, W) + k \cdot \sigma_{\text{error}}(t, W)$$
If $\text{Error}(t) > \tau_t$, flag an anomaly alert!

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 500

# Step 1: Simulate Normal Multivariate Telemetry (CPU, RAM, Network)
# Normal behavior: correlated metrics with noise
cpu = 30 + 10 * np.sin(np.linspace(0, 10, N)) + np.random.normal(0, 2, N)
ram = 40 + 5 * np.sin(np.linspace(0, 10, N)) + np.random.normal(0, 1, N)
network = 100 + 20 * np.sin(np.linspace(0, 10, N)) + np.random.normal(0, 5, N)

# Inject Sudden Anomaly at index 320 to 335 (Cyberattack / Memory Leak)
cpu[320:335] += 45.0
network[320:335] += 90.0

telemetry = np.column_stack([cpu, ram, network])

# Step 2: Simulate Reconstruction Model (PCA-based Reconstruction)
# Fit on early normal data (first 250 points)
normal_train = telemetry[:250]
mean_vec = np.mean(normal_train, axis=0)
centered_train = normal_train - mean_vec

cov = np.cov(centered_train, rowvar=False)
eigenvals, eigenvecs = np.linalg.eigh(cov)
# Keep top 2 principal components
top_vecs = eigenvecs[:, -2:]

# Reconstruct full dataset
centered_all = telemetry - mean_vec
projected = centered_all @ top_vecs
reconstructed = (projected @ top_vecs.T) + mean_vec

# Step 3: Compute Mean Squared Reconstruction Error
reconstruction_errors = np.mean((telemetry - reconstructed) ** 2, axis=1)

# Step 4: Dynamic Rolling Z-Score Threshold (Window = 40)
errors_series = pd.Series(reconstruction_errors)
rolling_mean = errors_series.rolling(window=40, min_periods=10).mean()
rolling_std = errors_series.rolling(window=40, min_periods=10).std()
dynamic_threshold = rolling_mean + 3.0 * rolling_std

# Identify Anomalies
detected_anomalies = np.where(reconstruction_errors > dynamic_threshold)[0]

print("--- Anomaly Detection Results ---")
print(f"True Injected Anomaly Window:  Index 320 to 334")
print(f"Detected Anomaly Indices:      {detected_anomalies.tolist()[:10]}...")
precision = len(set(detected_anomalies).intersection(set(range(320, 335)))) / max(1, len(detected_anomalies))
print(f"Detection Precision on Anomaly Window: {precision * 100:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Telemetry Correlation:** PCA learned the normal subspace correlation between CPU, RAM, and Network traffic.
2. **Reconstruction Failure:** When the abnormal spike occurred at index 320, the correlation broke and reconstruction error skyrocketed.
3. **Adaptive Threshold:** The rolling mean + $3\sigma$ threshold adapted to baseline variance, isolating the anomalous window with zero false alarms.

---

### 4. Industry Applications & Trade-offs
- **DevOps Datadog / AWS CloudWatch:** Automated root cause analysis for distributed microservice outages.
- **Trade-off:** If training data contains uncleaned anomalies, the model learns to reconstruct anomalies as "normal", reducing detection sensitivity.

---

### 5. Top Interview Questions & Answers
**Q1: Why is reconstruction probability preferred over deterministic reconstruction error in Variational Autoencoders (VAEs)?**
> **Answer:** Deterministic MSE treats all error directions equally. A VAE outputs mean $\mu$ and variance $\sigma^2$ per feature, calculating the log-likelihood $p(x \mid z)$. This accounts for features with naturally higher variance without triggering false alarms.

**Q2: What is the risk of using a fixed static threshold for time-series anomaly detection?**
> **Answer:** Fixed thresholds fail to account for concept drift, trend shifts, and periodic seasonality. A CPU usage of 80% might be completely normal during peak daytime business hours (false alarm) but an indicator of a severe runaway process at 3:00 AM on Sunday (missed detection).